In [ ]:
%pip install -U transformers datasets peft accelerate bitsandbytes

In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

# %cd /kaggle/working/Active-Reading--Pattern-Recognition # for kaggle env
%cd /content/Active-Reading--Pattern-Recognition # for colab env

os.getcwd()

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#test

Cloning into 'Active-Reading--Pattern-Recognition'...
remote: Enumerating objects: 3192, done.
remote: Counting objects: 100% (583/583), done.
remote: Compressing objects: 100% (527/527), done.
remote: Total 3192 (delta 98), reused 406 (delta 56), pack-reused 2609 (from 1)
Receiving objects: 100% (3192/3192), 99.24 MiB | 14.67 MiB/s, done.
Resolving deltas: 100% (763/763), done.
Updating files: 100% (2282/2282), done.
/content/Active-Reading--Pattern-Recognition


In [ ]:
%cd /content/Active-Reading--Pattern-Recognition

/content/Active-Reading--Pattern-Recognition


In [ ]:
!pip install flash-attn --no-build-isolation # for A100 when we use flash-attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 37.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from itertools import chain
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import concatenate_datasets


# --- Config ---
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
ORIGINAL_DATA_PATH = "/content/Active-Reading--Pattern-Recognition/Finetune_Datasets/financebench/original_mixin.jsonl"
DATA_PATH = "/content/Active-Reading--Pattern-Recognition/Finetune_Datasets/financebench/repetition_dataset.jsonl"
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4

# --- Dataset ---
repet_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
original_dataset = load_dataset("json", data_files=ORIGINAL_DATA_PATH, split="train")


dataset = concatenate_datasets([repet_dataset, original_dataset]).shuffle(seed=42)

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.float16, # for T4-GPU
    bnb_4bit_compute_dtype=torch.bfloat16, # for A100
    bnb_4bit_use_double_quant=True,
)

# --- Load Model (QLoRA) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2", # for A100
)

# --- Prep for k-bit training ---
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# --- LoRA ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- Tokenization ---
def tokenize_function(examples):
    return tokenizer(examples["text"])

def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH

    result = {
        k: [t[i:i + MAX_SEQ_LENGTH] for i in range(0, total_length, MAX_SEQ_LENGTH)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=2
)

lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    num_proc=2
)

# --- Training Args ---
training_args = TrainingArguments(
    output_dir="./qlora_qwen4b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LEARNING_RATE,
    max_steps=330,
    # fp16=True, # for T4-GPU
    bf16=True, # for A100
    tf32=True, # for A100
    logging_steps=10,
    save_steps=40, # changed from 5
    save_total_limit=2,
    report_to="none",
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    ),
)

print(" Starting QLoRA repetition fine-tuning...")
trainer.train(resume_from_checkpoint=False) # Change it to True after first run

print(" Saving adapter...")
model.save_pretrained("./final_qlora_adapter")
tokenizer.save_pretrained("./final_qlora_adapter")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Map (num_proc=2):   0%|          | 0/24 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/24 [00:00<?, ? examples/s]

 Starting QLoRA repetition fine-tuning...


Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss
5,1.637195
10,1.427886
15,1.390050
20,1.494951
25,1.516825
30,1.490811
35,1.368349
40,1.428319
45,1.372408
50,1.432508


 Saving adapter...


('./final_qlora_adapter/tokenizer_config.json',
 './final_qlora_adapter/chat_template.jinja',
 './final_qlora_adapter/tokenizer.json')

In [ ]:
!zip -r download.zip /content/Active-Reading--Pattern-Recognition/final_qlora_adapter

  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/ (stored 0%)
  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/chat_template.jinja (deflated 73%)
  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/adapter_config.json (deflated 58%)
  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/tokenizer.json (deflated 81%)
  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/README.md (deflated 65%)
  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/adapter_model.safetensors (deflated 7%)
  adding: content/Active-Reading--Pattern-Recognition/final_qlora_adapter/tokenizer_config.json (deflated 59%)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_PATH = "./final_qlora_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

In [ ]:
gen_kwargs = dict(
    max_new_tokens=128,
    do_sample=False,
    temperature=0.0,
)

def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

base_only_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base_only_model.eval()

prompts = [
    "Who received the IEEE Frank Rosenblatt Award in 2010?",
]

for p in prompts:
    print("=" * 80)
    print("PROMPT:", p)

    print("\nBASE:")
    print(generate(base_only_model, p))

    print("\nADAPTED:")
    print(generate(model, p))